# Conditional Strategies I: Timing Risk

The ideas are in the slides. Here you measure volatility from daily data, check whether it predicts returns, and build the
volatility-managed version of the market, of the other five factors, and of the best mix of all six.

**The plan**, one section per exercise in the slides

1. Realized variance from daily returns
2. 🎯 Sort the months by last month's volatility
3. 🎯 The volatility-managed market: your turn
4. All six factors
5. 🎯 Time the best mix. Then back to the slides
6. 🎯 Challenge (homework): one number and three sentences

📑 **Slides for this class:** [Conditional Strategies I (PDF)](https://amoreira2.github.io/UG54/slides/conditional_strategies_1.pdf)

<iframe src="../../slides/conditional_strategies_1.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]
import warnings; warnings.filterwarnings('ignore')

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

mkt = pd.read_parquet(f"{BASE}/factors_daily_1926.parquet")['Mkt-RF']   # daily market excess return, from 1926
D   = pd.read_parquet(f"{BASE}/ff6_daily_1963.parquet")      # daily excess returns on the six factors, from 1963

sharpe = lambda x: x.mean() / x.std() * np.sqrt(12)        # annualized Sharpe ratio of a monthly series

print(f"market:      {mkt.index[0]:%Y-%m-%d} to {mkt.index[-1]:%Y-%m-%d}, {len(mkt):,} days")
print(f"six factors: {D.index[0]:%Y-%m-%d} to {D.index[-1]:%Y-%m-%d}, {len(D):,} days")
print(D.columns.tolist())

---

## 1. Realized variance from daily returns

A month has about 21 trading days. Their spread tells you how risky the month was: add up the squared deviations of the
daily returns from their average. That is the month's **realized variance**. Square root, times $\sqrt{12}$, and you have a
volatility a year.

We need two things for each month: its return, and last month's realized variance, which you know before the month starts.

In [ ]:
daily = mkt                                                 # start with the market, 1926-2026

month = daily.index + pd.offsets.MonthEnd(0)                # label each day with the last day of its month
rv = daily.groupby(month).apply(lambda x: ((x - x.mean()) ** 2).sum())   # realized variance of each month
r  = (1 + daily).groupby(month).prod() - 1                  # the month's return: compound its days

M = pd.DataFrame({'r': r, 'rv': rv})
M['rv_lag'] = M['rv'].shift(1)                              # LAST month's variance: known when this month starts
M = M.dropna()

vol = np.sqrt(M['rv'] * 12)                                 # volatility, a year
vol.plot(title='realized volatility of the market, a year'); plt.show()

print(f"median volatility {vol.median():.1%}, highest {vol.max():.1%} in {vol.idxmax():%Y-%m}")
print(f"correlation of this month's volatility with last month's: {np.corrcoef(np.log(M.rv), np.log(M.rv_lag))[0, 1]:.2f}  (logs)")
print(f"correlation of this month's RETURN with last month's volatility: {np.corrcoef(M.r, M.rv_lag)[0, 1]:.3f}")

- Find the four highest spikes on the plot. Which events are they?
- One correlation is large and one is about zero. Which is which, and why is that the whole idea of today?
- Why `shift(1)`? What would go wrong without it?

---

## 2. 🎯 Sort the months by last month's volatility

Put the months in five groups, from the calmest last month to the most volatile, and look at what happened *next*.
The full sample, 1926–2026: the same months as the slides.

In [ ]:
group = pd.qcut(M['rv_lag'], 5, labels=['1 calm', '2', '3', '4', '5 volatile'])   # five equal groups of months

table = pd.DataFrame({
    'volatility next month': M['r'].groupby(group).std() * np.sqrt(12),
    'return next month':     M['r'].groupby(group).mean() * 12,
})
table['Sharpe ratio next month'] = table['return next month'] / table['volatility next month']
print(table.round(3).to_string())

table.plot.bar(subplots=True, layout=(1, 3), figsize=(13, 3.5), legend=False, rot=0); plt.show()

- After a volatile month, is the next month riskier? Is it more rewarding?
- If the return rose one for one with the variance, what would the Sharpe ratio column look like? What does it look like?
- Rerun the sort on `M.loc['1963':]`, the years of the next sections. Is the pattern as clear?

---

## 3. 🎯 The volatility-managed market

Hold less after a volatile month: the weight is $c / \text{RV}_{t-1}$. The constant $c$ only sets the scale. We pick it so
the managed strategy is exactly as volatile as the market, so the two can be compared directly.

From here on we use **1963–2026**, the years we have for all six factors, so the market can be compared with the others
in sections 4 and 5. `M63` is `M` from August 1963, the same months as those sections.

Three steps. The first is done for you.

1. `w_raw = 1 / M63['rv_lag']`
2. `c`: the market's volatility divided by the volatility of `w_raw * M63['r']`. Use `.std()` on each
3. `managed`: this month's return on the position, `c * w_raw * M63['r']`

In [ ]:
# === YOUR TURN ===
M63 = M.loc['1963-08':]           # the same months as sections 4-5: their data start July 1963, and July has no last month

w_raw = 1 / M63['rv_lag']         # step 1: less weight after a volatile month

c       = ____                    # step 2
managed = ____                    # step 3

In [ ]:
#@title 🔒 Check: the volatility-managed market
w_ref = 1 / M63['rv_lag']
c_ref = M63['r'].std() / (w_ref * M63['r']).std()
mgd   = c_ref * w_ref * M63['r']
weight = c_ref * w_ref                                      # the weight on the market each month

try:
    print("yours matches" if np.allclose(managed, mgd) else "yours differs: compare with the numbers below", "\n")
except Exception as e:
    print("yours: not built yet:", type(e).__name__, "\n")

fit = sm.OLS(mgd, sm.add_constant(M63['r'])).fit()         # managed return on the market's
print(f"market            Sharpe {sharpe(M63['r']):.2f}   volatility {M63['r'].std() * np.sqrt(12):.1%}")
print(f"managed           Sharpe {sharpe(mgd):.2f}   volatility {mgd.std() * np.sqrt(12):.1%}")
print(f"alpha on the market {fit.params['const'] * 12:.1%} a year, t = {fit.tvalues['const']:.1f}\n")

for a, b in [('1963', '1979'), ('1980', '2014'), ('2015', '2026')]:
    print(f"{a}-{b}: market {sharpe(M63['r'].loc[a:b]):.2f}, managed {sharpe(mgd.loc[a:b]):.2f}")

print(f"\nweight: median {weight.median():.2f}, largest {weight.max():.1f} in {weight.idxmax():%Y-%m}, above 2 in {np.mean(weight > 2):.0%} of months")
weight.plot(logy=True, title='weight on the market'); plt.axhline(1, color='k', lw=0.5); plt.show()

- Same volatility, so the Sharpe ratios can be compared directly. Did timing help the market since 1963?
- Look at the weights. What would you need to run this, and what would your risk manager say?

---

## 4. All six factors

The same three steps, in a loop over the six factors.

In [ ]:
rows = {}
for f in D.columns.drop('RF'):
    daily = D[f]
    month = daily.index + pd.offsets.MonthEnd(0)
    X = pd.DataFrame({'r':  (1 + daily).groupby(month).prod() - 1,
                      'rv': daily.groupby(month).apply(lambda x: ((x - x.mean()) ** 2).sum())})
    X['rv_lag'] = X['rv'].shift(1)
    X = X.dropna()

    w   = 1 / X['rv_lag']                                   # step 1
    c   = X['r'].std() / (w * X['r']).std()                 # step 2
    mgd = c * w * X['r']                                    # step 3
    fit = sm.OLS(mgd, sm.add_constant(X['r'])).fit()

    calm     = X['rv_lag'] <= X['rv_lag'].quantile(0.4)        # the calmest 40% of last months
    volatile = X['rv_lag'] >= X['rv_lag'].quantile(0.6)        # the most volatile 40%
    rows[f] = {'Sharpe': sharpe(X['r']), 'Sharpe managed': sharpe(mgd),
               'alpha': fit.params['const'] * 12, 't': fit.tvalues['const'],
               'Sharpe after calm': sharpe(X['r'][calm]), 'Sharpe after volatile': sharpe(X['r'][volatile])}

print(pd.DataFrame(rows).T.round(3).to_string())

- Which factors gain? Which lose?
- Look at the last two columns. What is different about the factors that lose? Connect it to the formula on the slides.

---

## 5. 🎯 Time the best mix

Section 4 timed each factor on its own. An investor who has read *Capital Allocation I* holds the best mix of all six. Does
timing still help once the factors already diversify each other?

1. The tangency weights: invert the covariance matrix of the six factors' monthly returns, times their means
2. The mix's **daily** return, so it has a realized variance of its own
3. The same three steps as section 3, on the mix

The weights use the whole sample, so the mix itself is in sample. We only ask whether timing it adds anything.

In [ ]:
factors = D.columns.drop('RF')
month = D.index + pd.offsets.MonthEnd(0)

# step 1: the best mix, from monthly returns: (covariance matrix)^-1 x means, then scaled
R = (1 + D[factors]).groupby(month).prod() - 1           # each factor's monthly return
cov = R.cov()                                             # covariance matrix, 6 x 6
W = np.linalg.inv(cov) @ R.mean()                         # the inverse times the mean returns
W = W / np.abs(W).sum()                                   # scale so the absolute weights add to one
print(pd.Series(W, index=factors).round(3).to_string(), "\n")

# step 2: the mix's daily return
mix = D[factors] @ W                                      # each day, the weighted sum of the six

# step 3: the same three steps as before
X = pd.DataFrame({'r':  (1 + mix).groupby(month).prod() - 1,
                  'rv': mix.groupby(month).apply(lambda x: ((x - x.mean()) ** 2).sum())})
X['rv_lag'] = X['rv'].shift(1)
X = X.dropna()
w   = 1 / X['rv_lag']
c   = X['r'].std() / (w * X['r']).std()
mgd = c * w * X['r']
fit = sm.OLS(mgd, sm.add_constant(X['r'])).fit()

print(f"best mix          Sharpe {sharpe(X['r']):.2f}")
print(f"best mix, managed Sharpe {sharpe(mgd):.2f}   alpha {fit.params['const'] * 12:.1%} a year, t = {fit.tvalues['const']:.1f}\n")
for a, b in [('1963', '1979'), ('1980', '2014'), ('2015', '2026')]:
    print(f"{a}-{b}: {sharpe(X['r'].loc[a:b]):.2f} -> {sharpe(mgd.loc[a:b]):.2f}")

- The mix already has a Sharpe ratio above 1. Does timing still add? How does the gain compare with the single factors?
- Which factors carry the mix? Given section 4, would you have expected timing to help it?
- The weights saw the whole sample. Why does that matter less for the timing question than for the mix itself?

---

## ⏸ Back to the slides

For the takeaways. The challenge below is homework.

---

## 🎯 Challenge: cap the leverage

*Homework — due before the next class. It should take ten minutes.*

The managed strategies above sometimes hold ten or more times the factor. Most investors cannot.

Take **momentum** (`UMD`), build the volatility-managed strategy exactly as in section 4, then cap the weight at 2:
after step 2, use `(c * w).clip(upper=2)` as the weight. Report the Sharpe ratio of the capped strategy, 1963–2026.

Then, in at most three sentences: did the cap cost anything? Why or why not?

> **📌 Required variable names:**
> ```python
> umd_cap2_sharpe = ____   # Sharpe ratio of volatility-managed UMD with the weight capped at 2
> MEMO = """..."""     # at most three sentences
> ```

In [ ]:
# Your work here


umd_cap2_sharpe = ____

print(f"capped at 2: Sharpe {umd_cap2_sharpe:.2f}")

In [ ]:
MEMO = """
Write your answer here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ['umd_cap2_sharpe', 'MEMO']
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L14_ConditionalStrategiesI_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")